# Co-Training EAGLE3 + LoRA for Cosmos-Reason2-8B

This notebook walks through the full workflow for **co-training** an EAGLE3 speculative-decoding
draft head **jointly with LoRA adapters** on [nvidia/Cosmos-Reason2-8B](https://huggingface.co/nvidia/Cosmos-Reason2-8B).

### Why co-train Eagle + LoRA?

Standard Eagle training freezes the base model and only trains the draft head.
By co-training a small LoRA adapter alongside the Eagle head, the base model’s
next-token distribution can become “easier” for the drafter to predict—improving
**acceptance rate (AR)** without significantly degrading model quality.

The combined loss naturally balances both objectives:
- **Eagle loss** (KL distillation): trains the draft head to predict the base model’s distribution
- **Base model loss** (cross-entropy): keeps the LoRA-adapted model on track with language modeling

### Workflow overview

| Step | Description |
| :---: | :--- |
| 1 | Install dependencies |
| 2 | Authenticate with Hugging Face |
| 3 | Prepare training data |
| 4 | Calibrate the draft vocabulary |
| 5 | Launch co-training |
| 6 | Export checkpoint for deployment |
| 7 | Deploy with vLLM |

> **Hardware requirement** – Cosmos-Reason2-8B co-training requires at least one 80 GB GPU
> (e.g. H100/A100). Because `eagle_freeze_base_model=False`, activations for the full base
> model are retained for backprop through LoRA, so memory usage is higher than standard
> Eagle-only training. Multi-GPU training is supported via FSDP2.

## Step 1 – Install Dependencies

In [ ]:
%%bash
pip install -U nvidia-modelopt[hf]
pip install -r ../requirements.txt
pip install peft

## Step 2 – Authenticate with Hugging Face

Both `nvidia/Cosmos-Reason2-8B` and `nvidia/Nemotron-Post-Training-Dataset-v2` require accepting
their licence agreements on the Hub. Run the cell below and follow the interactive prompt to log in:

In [ ]:
%%bash
hf auth login

## Step 3 – Prepare Training Data

We use a curated subset of [nvidia/Nemotron-Post-Training-Dataset-v2](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2)
(chat split) for training. The `nemotron_mapping.bin` file (bundled alongside this notebook) selects the specific rows to use.
It stores 0-based dataset row indices as packed `int32` values (little-endian, produced by `numpy.ndarray.tofile`).

The script streams only the required parquet shards and writes a conversation file in the
standard `jsonl` format expected by `launch_train_eagle_lora.sh`.

In [ ]:
%%bash
python ../prepare_input_conversations/add_nemotron_chat.py \
    --mapping-file nemotron_mapping.bin

In [ ]:
%%bash
# Expect exactly 89511 conversations.
count=$(wc -l < input_conversations/nemotron-chat.jsonl)
echo "${count} conversations in input_conversations/nemotron-chat.jsonl"
[ "$count" -eq 89511 ] || { echo "ERROR: expected 89511, got ${count}"; exit 1; }

## Step 4 – Calibrate the Draft Vocabulary

`CR2_eagle_config.json` sets `"draft_vocab_size": 32000`. Using a compressed vocabulary
speeds up training and inference, but requires a one-time calibration step that produces a
token-mapping file (`d2t.pt`).

In [ ]:
%%bash
python ../scripts/calibrate_draft_vocab.py \
    --model nvidia/Cosmos-Reason2-8B \
    --data input_conversations/nemotron-chat.jsonl \
    --draft_vocab_size 32000 \
    --save_dir draft_vocab_cache

## Step 5 – Launch Eagle3 + LoRA Co-Training

Training is launched via `launch_train_eagle_lora.sh`, which uses `accelerate launch`
and sets up FSDP2 automatically when multiple GPUs are available.

### Key arguments

| Argument | Value | Notes |
| :--- | :--- | :--- |
| `--model` | `nvidia/Cosmos-Reason2-8B` | Target VLM |
| `--data` | `guides/input_conversations/nemotron-chat.jsonl` | Training conversations |
| `--eagle_config` | `guides/CR2_eagle_config.json` | Draft-head architecture |
| `--draft_vocab_cache` | `guides/draft_vocab_cache/.../d2t.pt` | Token-mapping from Step 4 |
| `--vlm_processor` | `nvidia/Cosmos-Reason2-8B` | VLM image processor |
| `--vlm_img_dir` | `data/` | Directory containing referenced images |
| `--lora_r` | `16` | LoRA rank (higher = more capacity, more memory) |
| `--lora_alpha` | `32` | LoRA scaling factor |
| `--lora_dropout` | `0.05` | LoRA dropout |
| `--lora_target_modules` | `q_proj,k_proj,v_proj,o_proj` | Which layers get LoRA |
| `--training_seq_len` | `16384` | Max token length per sample |
| `--lr` | `1.5e-4` | Learning rate |
| `--num_epochs` | `20` | Training epochs |

### LoRA hyperparameter guidance

| Scenario | `lora_r` | `lora_alpha` | Notes |
| :--- | :--- | :--- | :--- |
| Conservative (minimal quality impact) | 8 | 16 | Few trainable params, subtle adaptation |
| Balanced (recommended starting point) | 16 | 32 | Good trade-off |
| Aggressive (max AR improvement) | 32–64 | 64–128 | More capacity, may need lower LR |

> **Tip** – Set `--ar_validate_steps` to a smaller value (e.g. `500`) to periodically measure
> acceptance rate on MT-Bench during training.

In [ ]:
%%bash
export WANDB_MODE=disabled
OUTPUT_DIR=ckpts/cosmos-reason2-8b-eagle3-lora
EAGLE_CONFIG=guides/CR2_eagle_config.json
DRAFT_VOCAB_CACHE=guides/draft_vocab_cache/Cosmos-Reason2-8B/d2t.pt

# 20 epochs on 89k samples (4xB100): ~30-36 hours (slightly longer than Eagle-only due to LoRA gradients).
cd ..; OUTPUT_DIR=$OUTPUT_DIR ./launch_train_eagle_lora.sh \
  --model nvidia/Cosmos-Reason2-8B \
  --output_dir $OUTPUT_DIR \
  --data guides/input_conversations/nemotron-chat.jsonl \
  --lr 1.5e-4 \
  --num_epochs 20 \
  --train_bs 1 \
  --eagle_config $EAGLE_CONFIG \
  --draft_vocab_cache $DRAFT_VOCAB_CACHE \
  --training_seq_len 16384 \
  --save_steps 1000 \
  --ar_validate_steps 1000000 \
  --vlm_processor nvidia/Cosmos-Reason2-8B \
  --vlm_img_dir data/ \
  --lora_r 16 \
  --lora_alpha 32 \
  --lora_dropout 0.05 \
  --lora_target_modules q_proj,k_proj,v_proj,o_proj

## Step 6 – Export Checkpoint for Deployment

After training, we export two artifacts:
1. **Eagle head checkpoint** – HuggingFace-compatible format for vLLM speculative decoding.
2. **Merged base model** – Original weights with LoRA adapters merged in.

Point `--model_path` to the desired checkpoint subdirectory (e.g. `checkpoint-110000`).

In [ ]:
%%bash
CKPT_DIR=ckpts/cosmos-reason2-8b-eagle3-lora/checkpoint-110000
EAGLE_CONFIG=guides/CR2_eagle_config.json
EAGLE_EXPORT=export/cosmos-reason2-8b-eagle3-lora-head
MERGED_EXPORT=export/cosmos-reason2-8b-lora-merged

python ../scripts/export_eagle_lora_checkpoint.py \
    --model_path $CKPT_DIR \
    --base_model nvidia/Cosmos-Reason2-8B \
    --eagle_config $EAGLE_CONFIG \
    --export_eagle_path $EAGLE_EXPORT \
    --export_merged_path $MERGED_EXPORT \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_target_modules q_proj,k_proj,v_proj,o_proj

## Step 7 – Deployment

Serve the LoRA-merged model with the Eagle3 head using **vLLM**:

```bash
vllm serve export/cosmos-reason2-8b-lora-merged \
    --host 0.0.0.0 \
    --port 8000 \
    --speculative_config '{"method": "eagle3", "model": "export/cosmos-reason2-8b-eagle3-lora-head", "num_speculative_tokens": 3}'
```

> **Important**: You must serve the **merged** model (not the original Cosmos-Reason2-8B),
> because the eagle head was co-trained with the LoRA-adapted distribution.
> Using the original model would result in a distribution mismatch and lower AR.

Refer to the [vLLM speculative decoding docs](https://docs.vllm.ai/en/latest/features/spec_decode/) for the full list of options.

## Comparison: Eagle-only vs Eagle + LoRA

| | Eagle-only (`train_eagle_head_cosmos_reason2.ipynb`) | Eagle + LoRA (this notebook) |
| :--- | :--- | :--- |
| Base model | Frozen (`torch.no_grad()`) | LoRA adapters trainable, rest frozen |
| Trainable params | Eagle head only (~300M) | Eagle head + LoRA (~300M + ~27M for r=16) |
| Loss | Eagle KL distillation only | Eagle KL + Base model CE |
| GPU memory | Lower (no base model gradients) | Higher (gradients through full base model) |
| Deployment | Original model + eagle head | LoRA-merged model + eagle head |
| Expected AR | Baseline | Higher (LoRA makes distribution easier to predict) |

## Appendix: How Co-Training Works

### Architecture (Cosmos-Reason2-8B)

```
Cosmos-Reason2-8B (Llama-3.1-8B based VLM)
├── visual (frozen)                    │ Vision encoder
├── model (language backbone)           │
│   ├── embed_tokens (frozen)           │
│   ├── layers.0–31                    │ Base model layers
│   │   ├── self_attn                   │
│   │   │   ├── q_proj (frozen)        │
│   │   │   │   └── + LoRA A/B (✓)     │ Trainable LoRA
│   │   │   ├── k_proj (frozen)        │
│   │   │   │   └── + LoRA A/B (✓)     │ Trainable LoRA
│   │   │   ├── v_proj (frozen)        │
│   │   │   │   └── + LoRA A/B (✓)     │ Trainable LoRA
│   │   │   └── o_proj (frozen)        │
│   │   │       └── + LoRA A/B (✓)     │ Trainable LoRA
│   │   └── mlp (frozen)               │
│   └── norm (frozen)                   │
├── lm_head (frozen)                    │
└── eagle_module (✓)                    │ Trainable draft head
    ├── fc (aux hidden state mixer)     │
    ├── layers.0 (LlamaDecoderLayer)    │
    ├── norm                            │
    └── lm_head (draft vocab 32k)       │
```

### Loss computation

```
Total Loss = Base Model CE Loss + Eagle KL Distillation Loss
             └──── LoRA adapts ───┘   └─── Eagle head learns ──┘
```

- **Base model CE loss** prevents the LoRA-adapted model from diverging.
- **Eagle KL distillation loss** trains the draft head to match the (LoRA-adapted) base model’s distribution.
- Gradients flow through the base model (needed for LoRA) but only LoRA and Eagle parameters update.

### Memory considerations

Co-training requires more memory than standard Eagle training because gradients must
flow through the base model (no `torch.no_grad()` wrapper). For Cosmos-Reason2-8B:

| Component | Approximate size |
| :--- | :--- |
| Base model weights (bf16) | ~16 GB |
| LoRA weights (r=16, 4 targets × 32 layers) | ~27 MB |
| Eagle head (draft vocab 32k) | ~300 MB |
| Activations & gradients (seq_len=16384) | ~40–60 GB |
| **Total** | **~60–80 GB** |

Reduce `--training_seq_len` or `--train_bs` if running low on GPU memory.
With 4× 80 GB GPUs and FSDP2, this fits comfortably.